# Évaluation Détaillée des Prénoms

Ce notebook présente une analyse complète des performances du modèle de regroupement des prénoms.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import evaluation_metrics as em

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Métriques de Performance (Précision / Rappel)

Évaluation externe basée sur un échantillon d'évaluation.

In [ ]:
eval_data = em.load_evaluation_data('./data/3_eval_prenoms.json')
metrics = em.compute_clustering_metrics(eval_data['paires_evaluation'])

print("--- RÉSULTATS ---")
print(f"Précision : {metrics['precision']:.2%}")
print(f"Rappel    : {metrics['recall']:.2%}")
print(f"F1-Score  : {metrics['f1']:.2%}")

em.plot_confusion_matrix(metrics['confusion_matrix'], title='Matrice de Confusion - Prénoms')

## 2. Degré de Précision d'après le Modèle (Confidence Score)

Représente la similarité moyenne des membres par rapport au centre de leur groupe (CamemBERT embeddings).

In [ ]:
conf_score = em.compute_confidence_scores('./data/3_eval_prenoms.json')
print(f"Degré de précision interne moyen (Modèle) : {conf_score:.4f} / 1.0000")

## 3. Cohérence Linguistique

Un groupe est considéré comme cohérent si la majorité de ses membres partagent la même origine linguistique.

In [ ]:
ratio, count = em.compute_language_consistency('./data/3_groupes_prenoms.json', './data/2_prenoms_clean.json')
print(f"Taux de cohérence linguistique : {ratio:.2%} (sur {count} groupes multi-membres)")

## 4. Analyse de la Taille des Groupes

Visualisation de la distribution pour détecter d'éventuels "méga-groupes".

In [ ]:
em.analyze_groups('./data/3_groupes_prenoms.json')

## 5. Analyse des Outliers

Focus sur les prénoms dont l'intégration au groupe est fragile.

In [ ]:
outliers = eval_data.get('outliers', [])
if outliers:
    print(f"Nombre de groupes contenant des outliers : {len(outliers)}")
    # Affichage des 5 plus gros groupes avec outliers
    for o in outliers[:5]:
        print(f"\nGroupe {o['groupe']} : {len(o['outliers'])} outliers détectés.")
        for p in o['outliers'][:3]:
            print(f"  - {p['prenom']} (sim={p['sim_centroid']:.4f})")
else:
    print("Aucun outlier détecté.")